# Method 2: MiniLM sentence embeddings + Logistic Regression
**Owner:** Mashael Alqahtani. Loads Arwa's train / dev / test splits.


In [4]:
from google.colab import drive
drive.mount("/content/drive")
!cp "/content/drive/MyDrive/splits.zip" .
!ls -la splits.zip

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw-r--r-- 1 root root 19262957 Nov 19 16:40 splits.zip


In [5]:
!unzip -q -o splits.zip      # creates data/splits/train.csv, dev.csv, test.csv

import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
SPLIT_DIR = Path("data/splits")      # same path as Arwa's Data section

train = pd.read_csv(SPLIT_DIR / "train.csv")
dev   = pd.read_csv(SPLIT_DIR / "dev.csv")
test  = pd.read_csv(SPLIT_DIR / "test.csv")

y_train, y_dev, y_test = train["label"].values, dev["label"].values, test["label"].values

for name, part in [("train", train), ("dev", dev), ("test", test)]:
    print(name, part.shape, part["label"].value_counts().to_dict())

train (27004, 5) {0: 14828, 1: 12176}
dev (5787, 5) {0: 3178, 1: 2609}
test (5787, 5) {0: 3178, 1: 2609}


## Sentence embeddings
Each text is converted by the pretrained all-MiniLM-L6-v2 model into 384 numbers. Results are saved as .npy files so they are computed only once.

In [6]:
!pip install -q sentence-transformers

import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

EMB_DIR = Path("data/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

def get_embeddings(texts, name):
    """Encode texts with MiniLM; load from disk if already computed."""
    path = EMB_DIR / f"{name}.npy"
    if path.exists():
        print(f"{name}: loaded from {path}")
        return np.load(path)
    X = model.encode(list(texts), batch_size=128, show_progress_bar=True)
    np.save(path, X)
    print(f"{name}: computed {X.shape}")
    return X

# Original text (training + tuning + first evaluation)
X_train = get_embeddings(train["input"], "train_input")
X_dev   = get_embeddings(dev["input"],   "dev_input")
X_test  = get_embeddings(test["input"],  "test_input")

# Cue-removed text
X_test_clean  = get_embeddings(test["input_clean"],  "test_clean")
X_train_clean = get_embeddings(train["input_clean"], "train_clean")
X_dev_clean   = get_embeddings(dev["input_clean"],   "dev_clean")

Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (16 outputs left out)

Batches:   0%|          | 0/211 [00:00<?, ?it/s]

train_clean: computed (27004, 384)


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

dev_clean: computed (5787, 384)


## Tuning C and testing
C is chosen on the dev set only. The model is then tested once on the same test examples, original and cue-removed.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

C_GRID = [0.01, 0.1, 1, 3, 10, 30, 100, 300, 1000]

def tune_C(X_tr, y_tr, X_dv, y_dv):
    """Pick C with the highest macro F1 on the dev set."""
    scores = {}
    for C in C_GRID:
        clf = LogisticRegression(C=C, max_iter=2000, random_state=SEED)
        clf.fit(X_tr, y_tr)
        scores[C] = f1_score(y_dv, clf.predict(X_dv), average="macro")
        print(f"C={C:<6} dev macro F1 = {scores[C]:.4f}")
    best_C = max(scores, key=scores.get)
    print("Best C:", best_C)
    return best_C

def evaluate(y_true, y_pred):
    return {"accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_F1": round(f1_score(y_true, y_pred, average="macro"), 4)}

# 1) Tune on dev (original text)
best_C = tune_C(X_train, y_train, X_dev, y_dev)

# 2) Train once with best C, test on the SAME examples: original vs cue-removed
m2 = LogisticRegression(C=best_C, max_iter=2000, random_state=SEED)
m2.fit(X_train, y_train)

pred_orig  = m2.predict(X_test)
pred_clean = m2.predict(X_test_clean)

results_m2 = pd.DataFrame({
    "Original test":    evaluate(y_test, pred_orig),
    "Cue-removed test": evaluate(y_test, pred_clean),
}).T
results_m2.loc["Drop"] = results_m2.loc["Original test"] - results_m2.loc["Cue-removed test"]
results_m2.index.name = "Method 2: MiniLM + LR"

Path("results").mkdir(exist_ok=True)
results_m2.to_csv("results/method2_results.csv")
print()
print(results_m2)
print()
print(classification_report(y_test, pred_clean, target_names=["Real", "Fake"], digits=4))

C=0.01   dev macro F1 = 0.8798
C=0.1    dev macro F1 = 0.9219
C=1      dev macro F1 = 0.9462
C=3      dev macro F1 = 0.9514
C=10     dev macro F1 = 0.9509
C=30     dev macro F1 = 0.9563
C=100    dev macro F1 = 0.9567
C=300    dev macro F1 = 0.9567
C=1000   dev macro F1 = 0.9555
Best C: 100

                       accuracy  macro_F1
Method 2: MiniLM + LR                    
Original test            0.9566    0.9562
Cue-removed test         0.9419    0.9415
Drop                     0.0147    0.0147

              precision    recall  f1-score   support

        Real     0.9540    0.9396    0.9467      3178
        Fake     0.9277    0.9448    0.9362      2609

    accuracy                         0.9419      5787
   macro avg     0.9409    0.9422    0.9415      5787
weighted avg     0.9422    0.9419    0.9420      5787



## Error analysis
Wrong predictions on the cue-removed test set, sorted by model confidence.

In [8]:
label_name = {0: "Real", 1: "Fake"}
prob_fake = m2.predict_proba(X_test_clean)[:, 1]

err = test.copy()
err["predicted"] = pred_clean
err["prob_fake"] = prob_fake
err["confidence"] = np.where(pred_clean == 1, prob_fake, 1 - prob_fake)
err = err[err["predicted"] != err["label"]]

print("Errors:", len(err), "of", len(test), f"({len(err)/len(test):.1%})")
print("Real predicted as Fake:", int((err.label == 0).sum()))
print("Fake predicted as Real:", int((err.label == 1).sum()))

err.sort_values("confidence", ascending=False).to_csv("results/method2_errors.csv", index=False)

def show(rows, title):
    print(f"\n========== {title} ==========")
    for _, r in rows.iterrows():
        print(f"\n[{r.id}] True: {label_name[r.label]} | Predicted: {label_name[r.predicted]} "
              f"| P(fake) = {r.prob_fake:.2f}")
        print("  ", r.input_clean[:300], "...")

top = err.sort_values("confidence", ascending=False)
show(top[top.label == 0].head(3), "Most confident: Real predicted as Fake")
show(top[top.label == 1].head(3), "Most confident: Fake predicted as Real")

Errors: 336 of 5787 (5.8%)
Real predicted as Fake: 192
Fake predicted as Real: 144

========== Most confident: Real predicted as Fake ==========

[real_1711] True: Real | Predicted: Fake | P(fake) = 1.00
   Trump on Twitter (Sept 15): London attack, Chain migration, ESPN, U.S. Air Force. : - With the ridiculous Filibuster Rule in the Senate, Republicans need 60 votes to pass legislation, rather than 51. Can’t get votes, END NOW![0637 EDT] - Another attack in London by a loser terrorist. These are sick  ...

[real_2275] True: Real | Predicted: Fake | P(fake) = 1.00
   Trump on Twitter (Aug 7): Fake News, Senator Richard Blumenthal. : - The failing , which has made every wrong prediction about me including my big election win (apologized), is totally inept! [0638 EDT] - The Trump base is far bigger & stronger than ever before (despite some phony Fake News polling) ...

[real_17110] True: Real | Predicted: Fake | P(fake) = 1.00
   Khamenei says Iran will 'shred' nuclear deal if U.S. quits

In [9]:
import re
N_WORDS = 200

_DATELINE   = re.compile(r"^[^\n]{0,150}?\(\s*reuters\s*\)\s*[-\u2013\u2014]*\s*", re.I)
_URL        = re.compile(r"(https?://\S+|www\.\S+|pic\.twitter\.com/\S+|\bbit\.ly/\S+|\bt\.co/\S+)", re.I)
_HANDLE     = re.compile(r"@\w+")
_IMG_CREDIT = re.compile(r"(featured image|header image|images?|photos?|screenshot|screen capture|screengrab)"
                         r"\s*(via|by|credit|credits|courtesy of|from)\b[^.\n]*", re.I)
_SITE_NAMES = re.compile(r"getty\s*images|\b(?:21st century wire|21wire|reuters)\b", re.I)
_MEDIA_TAG  = re.compile(r"[\(\[]\s*(video|videos|tweet|tweets|watch|image|images|photo|photos|audio|"
                         r"screenshot|screenshots|details|graphic video)\s*[\)\]]", re.I)
# Reuters note on "Trump on Twitter" articles; matched in 3 parts with \s+ so extra or
# non-breaking spaces between words do not stop the match
_TWEET_NOTE = [
    re.compile(r"the\s+following\s+statements?\s+(?:was|were)\s+posted\s+to\s+the\s+verified\s+"
               r"twitter\s+accounts?\s+of\s+(?:u\.\s*s\.\s+)?(?:president(?:-elect)?\s+)?donald\s+(?:j\.\s+)?trump"
               r"(?:\s*,?\s*(?:and\s+)?@\w+)*\s*\.?", re.I),
    re.compile(r"the\s+opinions?\s+expressed\s+(?:are|is)\s+(?:his|her|their)\s+own\s*\.?", re.I),
    re.compile(r"(?:reuters\s+)?has\s+not\s+edited\s+the\s+statements?\s+or\s+confirmed\s+(?:their|its)\s+accuracy\s*\.?", re.I),
]
_CREDITS    = re.compile(r"\(\s*(?:additional\s+)?(?:reporting|writing|editing|compiled|this story|this version)\b[^)]*\)", re.I)
_EMPTY_PAR  = re.compile(r"\(\s*\)|\[\s*\]")
_SPACES     = re.compile(r"\s+")

def remove_cues(text: str) -> str:
    """Remove source-identifying cues from a title or article body.
    Repeated until nothing changes, because removing one cue can bring two
    parts of another cue together, e.g. "Image (VIDEO) via YouTube"."""
    if not isinstance(text, str):
        return ""
    t = _DATELINE.sub("", text)
    for _ in range(5):
        before = t
        for rx in _TWEET_NOTE:
            t = rx.sub(" ", t)
        t = _CREDITS.sub(" ", t)
        t = _URL.sub(" ", t)
        t = _HANDLE.sub(" ", t)
        t = _IMG_CREDIT.sub(" ", t)
        t = _MEDIA_TAG.sub(" ", t)
        t = _SITE_NAMES.sub(" ", t)
        t = _EMPTY_PAR.sub(" ", t)
        t = _SPACES.sub(" ", t)
        if t == before:
            break
    return _SPACES.sub(" ", t).strip()

def first_words(text: str, n: int = N_WORDS) -> str:
    return " ".join(str(text).split()[:n])

def make_input(title: str, body: str) -> str:
    """Model input = title + first N_WORDS words of the body (same for both methods)."""
    return (str(title).strip() + ". " + first_words(body)).strip()

# quick checks that the rules do what we expect
tests = {
    "WASHINGTON (Reuters) - The Senate voted on Tuesday.": "The Senate voted on Tuesday.",
    "LONDON/PARIS (Reuters) -  Talks resumed.": "Talks resumed.",
    "He told Reuters the plan was final.": "He told the plan was final.",
    "Watch here https://t.co/abc123 pic.twitter.com/XyZ now": "Watch here now",
    "— Donald J. Trump (@realDonaldTrump) May 5, 2017": "— Donald J. Trump May 5, 2017",
    "The end. Featured image via Getty Images": "The end.",
    "Photo by Chip Somodevilla/Getty Images. Next sentence.": ". Next sentence.",
    "Trump SLAMS reporter (VIDEO)": "Trump SLAMS reporter",
    "21st Century Wire says the vote was rigged": "says the vote was rigged",
    ("The following statements were posted to the verified Twitter accounts of U.S. President Donald Trump, "
     "@realDonaldTrump and @POTUS. The opinions expressed are his own. Reuters has not edited the statements "
     "or confirmed their accuracy. @realDonaldTrump : - Tax cuts"): ": - Tax cuts",
    "Prices rose. (Reporting by Jane Doe; Editing by Tom Brown)": "Prices rose.",
    ("The following  statements were posted to the verified Twitter accounts of U.S. President Donald Trump, "
     "@realDonaldTrump and @POTUS.  The opinions expressed are his own. Reuters has not edited the  statements "
     "or confirmed their accuracy.  @realDonaldTrump : - Tax cuts"): ": - Tax cuts",
    "The Federal Reserve kept rates unchanged.": "The Federal Reserve kept rates unchanged.",
}
for raw_text, expected in tests.items():
    got = remove_cues(raw_text)
    assert got == expected, f"\n input:    {raw_text!r}\n expected: {expected!r}\n got:      {got!r}"
print(f"All {len(tests)} remove_cues checks passed.")

All 13 remove_cues checks passed.


## Final Try-it model
Retrained on cue-removed text and saved, so new text can be classified without retraining.

In [10]:
import joblib

# 1) Tune C again, this time on cue-removed text
print("Tuning on cue-removed text:")
best_C_clean = tune_C(X_train_clean, y_train, X_dev_clean, y_dev)

# 2) Final model: trained on cue-removed text
m2_final = LogisticRegression(C=best_C_clean, max_iter=2000, random_state=SEED)
m2_final.fit(X_train_clean, y_train)

final_scores = evaluate(y_test, m2_final.predict(X_test_clean))
print("\nFinal model (trained on clean) on cue-removed test:", final_scores)

# 3) Save it so the Try-it cell needs no retraining
Path("models").mkdir(exist_ok=True)
joblib.dump(m2_final, "models/method2_minilm_lr.joblib")
print("Saved: models/method2_minilm_lr.joblib")

# 4) Prediction function for new text
def predict_m2(text, title=""):
    """Classify a new news text as Fake or Real with Method 2."""
    if text is None or not str(text).strip():
        return "Please enter a news text (it was empty)."
    clean = make_input(remove_cues(title), remove_cues(text))
    emb = model.encode([clean])
    p_fake = m2_final.predict_proba(emb)[0, 1]
    label = "Fake" if p_fake >= 0.5 else "Real"
    return f"{label}  (P(fake) = {p_fake:.2f})"

# Quick test (examples from the proposal + empty input)
print(predict_m2("Pope Francis endorses Donald Trump for president, shocking the world"))
print(predict_m2("The Federal Reserve kept interest rates unchanged on Wednesday, citing steady job growth"))
print(predict_m2("   "))

Tuning on cue-removed text:
C=0.01   dev macro F1 = 0.8714
C=0.1    dev macro F1 = 0.9137
C=1      dev macro F1 = 0.9374
C=3      dev macro F1 = 0.9441
C=10     dev macro F1 = 0.9459
C=30     dev macro F1 = 0.9486
C=100    dev macro F1 = 0.9464
C=300    dev macro F1 = 0.9478
C=1000   dev macro F1 = 0.9469
Best C: 30

Final model (trained on clean) on cue-removed test: {'accuracy': 0.9485, 'macro_F1': 0.948}
Saved: models/method2_minilm_lr.joblib
Fake  (P(fake) = 0.95)
Real  (P(fake) = 0.00)
Please enter a news text (it was empty).


In [11]:
import joblib
m1 = joblib.load("method1_tfidf_lr.joblib")

rows = {}
for name, m_pred_o, m_pred_c in [
    ("TF-IDF + LR", m1.predict(test["input"]), m1.predict(test["input_clean"])),
    ("MiniLM + LR", pred_orig, pred_clean),
]:
    o, c = evaluate(y_test, m_pred_o), evaluate(y_test, m_pred_c)
    rows[name] = {"Original F1": o["macro_F1"], "Cue-removed F1": c["macro_F1"],
                  "Drop": round(o["macro_F1"] - c["macro_F1"], 4)}

comparison = pd.DataFrame(rows).T
comparison.to_csv("results/comparison.csv")
comparison

,Original F1,Cue-removed F1,Drop
TF-IDF + LR,0.9942,0.9670,0.0272
MiniLM + LR,0.9562,0.9415,0.0147


## Original examples (Mashael)
Two examples written by me, not from the dataset.

In [12]:
# Mashael's original examples (written by me, not from the dataset)
my_examples = [
    ("Real", "Saudi Arabia’s Ministry of Energy said on November 8 that an administrative building belonging to Aramco in Riyadh was the target of an attempted attack. Debris fell on the site and caused limited damage to the building, and no injuries were reported."),
    ("Fake", "BREAKING: Saudi Arabia’s “Madrasati” School Platform HACKED! Millions of Students’ Data Exposed. Insiders reveal that the Madrasati platform was secretly hacked on October 16, and officials are trying to cover it up. An unknown group is behind the attack, and nobody knows what they did with the data. You won’t believe how long this went unnoticed!"),
]

for true_label, text in my_examples:
    print(f"True: {true_label}  ->  Method 2: {predict_m2(text)}")
    print("   ", text, "\n")

True: Real  ->  Method 2: Real  (P(fake) = 0.01)
    Saudi Arabia’s Ministry of Energy said on November 8 that an administrative building belonging to Aramco in Riyadh was the target of an attempted attack. Debris fell on the site and caused limited damage to the building, and no injuries were reported. 

True: Fake  ->  Method 2: Fake  (P(fake) = 0.52)
    BREAKING: Saudi Arabia’s “Madrasati” School Platform HACKED! Millions of Students’ Data Exposed. Insiders reveal that the Madrasati platform was secretly hacked on October 16, and officials are trying to cover it up. An unknown group is behind the attack, and nobody knows what they did with the data. You won’t believe how long this went unnoticed! 

